# Weight Initialization

---

## 1. What is Weight Initialization?

Before training begins, every weight and bias in a neural network must be assigned a **starting value**. This is called weight initialization.

The choice of initial values has a profound impact on training:

- **Bad initialization** → vanishing/exploding gradients, slow convergence, or the network never learns at all
- **Good initialization** → gradients flow properly, activations stay in a healthy range, and training converges faster

> **Core goal of weight initialization:** Keep the signal (activations) and the gradients neither too large nor too small as they pass through the layers.

---

## 2. What NOT to Do — Common Mistakes

We use our familiar 2-layer network for all examples:

$$o_{11} = w^{1}_{11} x_1 + w^{1}_{21} x_2 + b_{11}, \qquad o_{12} = w^{1}_{12} x_1 + w^{1}_{22} x_2 + b_{12}$$

$$\hat{y} = w^{2}_{11} \cdot o_{11} + w^{2}_{21} \cdot o_{12} + b_{21}$$

With inputs $x_1 = 80$, $x_2 = 8$.

---

### Case 1 — Zero Initialization

Set all weights and biases to 0:

$$w^{1}_{11} = w^{1}_{21} = w^{1}_{12} = w^{1}_{22} = w^{2}_{11} = w^{2}_{21} = b_{11} = b_{12} = b_{21} = 0$$

**With ReLU activation:**

$$o_{11} = \text{ReLU}(0 \cdot 80 + 0 \cdot 8 + 0) = \text{ReLU}(0) = 0$$

$$o_{12} = \text{ReLU}(0 \cdot 80 + 0 \cdot 8 + 0) = \text{ReLU}(0) = 0$$

Both hidden neurons output 0. The output is also 0:

$$\hat{y} = 0 \cdot 0 + 0 \cdot 0 + 0 = 0$$

During backpropagation, the gradient flowing to the hidden layer is:

$$\frac{\partial L}{\partial w^{1}_{11}} = -2(y - \hat{y}) \cdot w^{2}_{11} \cdot x_1 = -2(y - 0) \cdot 0 \cdot 80 = 0$$

The gradient is **zero for every weight**. Nothing updates. Training fails completely.

**With Sigmoid activation:**

$$o_{11} = \sigma(0) = \frac{1}{1 + e^{0}} = 0.5, \qquad o_{12} = \sigma(0) = 0.5$$

Both neurons output the same value $0.5$. The gradients are also identical:

$$\frac{\partial L}{\partial w^{1}_{11}} = \frac{\partial L}{\partial w^{1}_{12}} = \frac{\partial L}{\partial w^{1}_{21}} = \frac{\partial L}{\partial w^{1}_{22}}$$

Since every neuron gets the same gradient, every weight updates by the same amount. After the update, all weights are **still equal**. This repeats every epoch — the two hidden neurons remain **permanently identical**.

This is called the **symmetry problem**. The two neurons in the hidden layer are indistinguishable — the network effectively behaves as if it only has **1 hidden neuron**, wasting the entire layer.

> **Zero initialization always fails** — either the network is completely stuck (ReLU) or it loses all representational capacity (sigmoid/tanh).

---

### Case 2 — Constant Non-Zero Initialization

Set all weights to the same non-zero constant, e.g. $w = 1$, $b = 0$:

**With ReLU activation:**

$$o_{11} = \text{ReLU}(1 \cdot 80 + 1 \cdot 8 + 0) = 88$$

$$o_{12} = \text{ReLU}(1 \cdot 80 + 1 \cdot 8 + 0) = 88$$

Non-zero outputs — so the network is not dead. But the symmetry problem remains:

$$o_{11} = o_{12} = 88$$

Both neurons compute the exact same thing. Their gradients are identical. Their updates are identical. Layer after layer, every neuron remains a clone of every other neuron in its layer.

No matter how many neurons you add, the network behaves like a **single-neuron-per-layer model** — it can only learn linear functions and loses all the power of depth and width.

This is why non-zero constant initialization is also not viable.

> **The symmetry problem** is the fundamental issue with both zero and constant initialization. For a network to learn diverse features, neurons must **start differently** so they can **evolve differently**.

---

### Case 3 — Small Random Numbers

Initialize weights with very small random values, e.g.:

$$w \sim \mathcal{U}(-0.01,\ 0.01) \quad \text{or} \quad w \sim \mathcal{N}(0,\ 0.01)$$

This breaks symmetry — neurons start differently and will evolve differently. But using values that are **too small** creates a new problem.

**With Sigmoid activation — vanishing gradients:**

For small weights, the pre-activation $z = w \cdot x + b$ is very close to 0 for all neurons. The sigmoid derivative at $z \approx 0$ is:

$$\sigma'(0) = 0.25 \quad \text{(the maximum, but still less than 1)}$$

As the signal passes through many layers, it gets multiplied by $\approx 0.25$ repeatedly. By layer 5 or 10, the gradient is microscopic — **vanishing gradients** take over.

**With ReLU activation — approaching a linear model:**

With very small weights, the pre-activations are small and positive, so ReLU passes them through. But after several layers, the outputs are so small that the network behaves approximately **linearly** — it loses the non-linear representation power that makes deep networks useful.

The problem is not random initialization itself — it is **poorly scaled** random initialization. The fix is to choose the right **scale** for the random values, which is exactly what Xavier and He initialization do.

---

## 4. Summary

| Initialization | Symmetry broken? | Gradient flow | Use with |
|---|---|---|---|
| **Zero** | ❌ | ❌ Completely blocked (ReLU) or symmetric (sigmoid) | Never |
| **Constant non-zero** | ❌ | ⚠️ Flows but all neurons stay identical | Never |
| **Small random** | ✅ | ⚠️ Vanishes over depth | Avoid for deep networks |
---

## 5. Key Takeaways

- Weight initialization must **break symmetry** — if neurons start identical, they stay identical and the network wastes capacity.
- **Zero and constant initialization** always fail: either gradients are blocked entirely or the symmetry problem makes the whole layer equivalent to one neuron.
- **Small random values** break symmetry but are poorly scaled — they cause vanishing gradients in deep networks.